# 5. Cross-recurrence (cRQA) between pairs of participants

**Run once per session.** Input: the envelopes of `04_spatial_filter` (one per participant and area, 25 Hz). Output: one table with %REC and %DET for every **pair × trial × area**, with the pair's **role** in that trial, plus chance levels from surrogate pairs.

**Design A** (`CLAUDE.md`): the same pair is compared across the roles it had: `teammates` / `opponents` (Team trials), `solo` (Solo trials, playing at the same time without interacting), `baseline` (motor baseline: the same bubbles on all iPads).

**Measures near lag 0** (decided 2026-11-19). The cross-recurrence plot compares person A at every moment with person B at every moment of the trial. Over the whole plot, %REC and %DET cannot tell coupled from unrelated envelopes (synthetic test: a signal paired with itself changed %REC from 5.0% to 5.1%), because synchrony shows only near the line of synchronisation (lag 0) and smooth envelopes make diagonal lines anyway. So, following the diagonal profile of Coco et al. (2021):
- %REC and %DET are taken on the diagonals with |lag| ≤ `CRQA_LAG_BAND_S` (0.5 s);
- %DET counts lines of at least `CRQA_MIN_LINE_S` (0.4 s = 10 samples), not 2 samples (which gave ~98% for anything);
- the full profile (%REC per lag, −5 to +5 s) is saved, so a peak at a longer lag would still be visible.

**Surrogate pairs** give the chance level: A's trial against B's *other* trial of the same condition (same pair, same kind of play, but not at the same time). Coupling that also shows in the surrogates comes from similar envelopes, not from playing together.

**Steps:** 1. envelopes and pairs → 2. trials and roles → 3. cut the windows in session time → 4. parameters (once per area) → 5. cRQA (real and surrogate) → 6. save → 7. overview.

Functions: `crqa.py`. Settings: `config.py` (`CRQA_*`).

In [ ]:
import itertools
import os

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config as cfg  # also puts rt_bids on the path
import crqa
import rt_bids
%matplotlib inline
mne.set_log_level("error")

## 0. Session

In [ ]:
# execute_notebooks.py passes the session in RT_SUBJECT; when running by hand, change the default here
SESSION = os.environ.get("RT_SUBJECT", "session03")
OUT = cfg.CRQA_ROOT / SESSION
OUT.mkdir(parents=True, exist_ok=True)
FS = cfg.ENVELOPE_SFREQ
print(SESSION, "->", OUT)

## 1. Envelopes and pairs

For every participant of the session with a choice in `SPATIAL_FILTER_CHOICE`, the envelopes saved by `04` are loaded. What `04` actually used is read from `<sub>_spatial_filter.csv` and compared with `config.py`: if they differ, `04` has to be rerun first.

A pair is two participants with the same area (e.g. both have a `visual` envelope). Every pair is used once (A = the lower participant number).

In [ ]:
subjects = [s for s in cfg.session_subjects(SESSION) if s in cfg.SPATIAL_FILTER_CHOICE]
envelopes, rows = {}, []
first_samp, clean_sfreq = {}, {}
for sub in subjects:
    used = pd.read_csv(cfg.out_dir(sub) / f"{sub}_spatial_filter.csv")
    for r in used.itertuples():
        wanted = cfg.SPATIAL_FILTER_CHOICE[sub].get(r.area)
        assert wanted == (r.ica, r.component, r.band), (
            f"{sub} {r.area}: config.py says {wanted}, 04 used {(r.ica, r.component, r.band)}: rerun 04")
        raw = mne.io.read_raw(cfg.out_dir(sub) / f"{sub}_{r.area}_{r.band}_{r.ica}_envelope_raw.fif", preload=True)
        assert raw.info["sfreq"] == FS, f"{sub} {r.area}: envelope at {raw.info['sfreq']} Hz, expected {FS}"
        envelopes[(sub, r.area)] = {"raw": raw, "x": raw.get_data()[0], "band": r.band, "ica": r.ica,
                                    "component": int(r.component)}
        rows.append({"subject": sub, "area": r.area, "ica": r.ica, "component": r.component, "band": r.band})
    # the clean data's first sample (500 Hz): the envelope starts there (01 cut the recording before it)
    clean = mne.io.read_raw(cfg.out_dir(sub) / f"{sub}_clean_raw.fif", preload=False)
    first_samp[sub], clean_sfreq[sub] = clean.first_samp, clean.info["sfreq"]

areas = sorted({area for (_, area) in envelopes})
pairs = [(a, b, area) for area in areas
         for a, b in itertools.combinations(sorted(s for (s, ar) in envelopes if ar == area), 2)]
print(f"{len(subjects)} participants with envelopes; areas: {areas}; {len(pairs)} pair x area combinations")
pd.DataFrame(rows)

## 2. Trials and roles

The trials come from the behavioural events of the session (`rt_bids.session_events`, `trial_type == "trial"`), in **session time** (the clock all recordings share). Each person has a row per trial with its start, duration and team (`team_id`, a fact about the trial: teams are reassigned between blocks).

For each pair and trial, the **window both were playing** is used: from the later start to the earlier end. The role:
- `joint` trial, same `team_id` → `teammates`; different → `opponents`;
- `individual` → `solo`; `motor_baseline` → `baseline`.

In [ ]:
events = rt_bids.session_events(cfg.BIDS_ROOT, SESSION)
trials = events[(events["trial_type"] == "trial") & events["condition"].isin(cfg.CONDITIONS)]
trials = trials[["participant_id", "condition", "round", "trial", "team_id", "onset", "duration"]].copy()
trials["end"] = trials["onset"] + trials["duration"]


def role(condition, team_a, team_b):
    if condition == "joint":
        return "teammates" if team_a == team_b else "opponents"
    return {"individual": "solo", "motor_baseline": "baseline"}[condition]


windows = []
for a, b in sorted({(a, b) for a, b, _ in pairs}):
    both = trials[trials["participant_id"] == a].merge(
        trials[trials["participant_id"] == b], on=["condition", "round", "trial"], suffixes=("_a", "_b"))
    for t in both.itertuples():
        windows.append({"participant_a": a, "participant_b": b, "condition": t.condition, "round": int(t.round),
                        "trial": int(t.trial), "team_a": t.team_id_a, "team_b": t.team_id_b,
                        "role": role(t.condition, t.team_id_a, t.team_id_b),
                        "start": max(t.onset_a, t.onset_b), "end": min(t.end_a, t.end_b)})
windows = pd.DataFrame(windows)
print(windows.groupby(["participant_a", "participant_b", "role"]).size().unstack(fill_value=0))

## 3. Cutting the windows in session time

Session time → sample in the envelope, per person:
1. `rt_bids.from_session_time(<recording>, t)` gives the (fractional) sample in the **original** recording at 500 Hz. It corrects for each headset's clock drift against the session clock and for skipped samples (`acq_gap`).
2. The envelope starts at the clean data's first sample (`first_samp`; `01` cut the recording 2 s before the first trial) and has 25 samples per second instead of 500: envelope sample = (original sample − `first_samp`) × 25 / 500.

(`rt_bids.crop_session` is not used: it ignores `first_samp`, so it is wrong on our cropped files.)

Both cuts start at the first whole sample inside the window; the longer one is trimmed to the shorter (they can differ by a sample). The share of each window marked bad (`BAD_amplitude` from `01`) is recorded, not removed, so the analysis can decide.

In [ ]:
def envelope_samples(sub, times):
    # fractional envelope sample(s) of session time(s)
    k = rt_bids.from_session_time(cfg.bdf_path(sub), times)          # original recording, 500 Hz
    return (k - first_samp[sub]) * FS / clean_sfreq[sub]


def bad_fraction(raw, start, stop):
    # share of the samples start..stop (envelope samples) covered by annotations starting with BAD_
    ann = raw.annotations
    offset = raw.first_time if ann.orig_time is not None else 0.0
    bad = np.zeros(stop - start, bool)
    for onset, duration, description in zip(ann.onset, ann.duration, ann.description):
        if description.startswith("BAD_"):
            i0 = int(np.floor((onset - offset) * FS)) - start
            i1 = int(np.ceil((onset - offset + duration) * FS)) - start
            bad[max(i0, 0):max(min(i1, len(bad)), 0)] = True
    return float(bad.mean())


segments = []  # one per pair x area x window: the two cut envelopes
for w in windows.itertuples():
    ia = envelope_samples(w.participant_a, [w.start, w.end])
    ib = envelope_samples(w.participant_b, [w.start, w.end])
    if np.isnan(ia).any() or np.isnan(ib).any():
        print(f"skipped (outside a recording or in a gap): {w.participant_a}-{w.participant_b} {w.condition} r{w.round} t{w.trial}")
        continue
    sa, sb = int(np.ceil(ia[0])), int(np.ceil(ib[0]))
    n = min(int(np.floor(ia[1])) - sa, int(np.floor(ib[1])) - sb)
    for area in areas:
        if (w.participant_a, area) not in envelopes or (w.participant_b, area) not in envelopes:
            continue
        ea, eb = envelopes[(w.participant_a, area)], envelopes[(w.participant_b, area)]
        assert sa >= 0 and sb >= 0 and sa + n <= len(ea["x"]) and sb + n <= len(eb["x"]), "window outside the envelope"
        info = {k: v for k, v in w._asdict().items() if k != "Index"}
        segments.append({**info, "area": area, "n_samples": n,
                         "x_a": ea["x"][sa:sa + n], "x_b": eb["x"][sb:sb + n],
                         "bad_fraction_a": bad_fraction(ea["raw"], sa, sa + n),
                         "bad_fraction_b": bad_fraction(eb["raw"], sb, sb + n)})

seg_table = pd.DataFrame([{k: v for k, v in s.items() if not k.startswith("x_")} for s in segments])
seg_table["duration_s"] = seg_table["n_samples"] / FS
print(seg_table.groupby(["area", "role"]).agg(n=("n_samples", "size"), duration_s=("duration_s", "median"),
                                              bad_max=("bad_fraction_a", "max")).round(3))

## 4. Parameters, chosen once per area

The embedding (delay τ, dimension m) and the radius are chosen **once per area** (alpha and beta envelopes change at different speeds) from all windows of all pairs of the session, with all roles pooled, so no role is favoured. Then they are used for every window, real and surrogate.

- **τ:** for every envelope window, the first minimum of the auto mutual information (AMI) within `CRQA_AMI_MAX_LAG_S`; the median over windows.
- **m:** for every envelope window, the smallest m with at most `CRQA_FNN_THRESHOLD` (10%) false nearest neighbours; the median over windows.
- **Radius:** for every pair window, the radius at which `CRQA_TARGET_REC` (5%) of all cross-recurrence points are recurrent; the median over windows. It is **fixed**: tuning it per window would make %REC the same everywhere.

In [ ]:
params = {}
for area in areas:
    segs = [s for s in segments if s["area"] == area]
    # every envelope window once (a participant's window is shared by all their pairs)
    unique = {}
    for s in segs:
        for who, x in (("participant_a", s["x_a"]), ("participant_b", s["x_b"])):
            unique[(s[who], s["condition"], s["round"], s["trial"])] = x
    taus = [crqa.first_minimum(crqa.auto_mutual_information(x, int(cfg.CRQA_AMI_MAX_LAG_S * FS))) for x in unique.values()]
    tau = int(np.median(taus))
    dims = [crqa.choose_dimension(crqa.false_nearest_neighbours(x, tau, cfg.CRQA_MAX_DIM), cfg.CRQA_FNN_THRESHOLD)
            for x in unique.values()]
    m = int(np.ceil(np.median(dims)))
    radii = [crqa.radius_for_rec(s["x_a"], s["x_b"], m, tau, cfg.CRQA_TARGET_REC, seed=cfg.RANDOM_STATE) for s in segs]
    params[area] = {"area": area, "tau": tau, "tau_s": tau / FS, "m": m, "radius": float(np.median(radii)),
                    "tau_range": f"{min(taus)}-{max(taus)}", "m_range": f"{min(dims)}-{max(dims)}",
                    "radius_range": f"{min(radii):.3f}-{max(radii):.3f}", "n_envelope_windows": len(unique),
                    "lag_band_s": cfg.CRQA_LAG_BAND_S, "min_line_s": cfg.CRQA_MIN_LINE_S,
                    "target_rec": cfg.CRQA_TARGET_REC}
params_table = pd.DataFrame(params.values())
params_table.to_csv(OUT / f"crqa_parameters_{SESSION}.csv", index=False)
params_table

## 5. cRQA: real pairs and surrogate pairs

For every window: %REC and %DET in the lag band (|lag| ≤ 0.5 s), the mean and longest line (s), and the diagonal profile (%REC per lag, −5 to +5 s).

**Surrogates:** for each pair, area and condition with more than one trial (`joint`, `individual`), A's trial *i* against B's trial *j* ≠ *i* (trimmed to the shorter). They get `role = "surrogate"` and keep the condition, so each role can be compared with the chance level of its own kind of play.

In [ ]:
band = int(round(cfg.CRQA_LAG_BAND_S * FS))
min_line = int(round(cfg.CRQA_MIN_LINE_S * FS))
profile_lag = int(round(cfg.CRQA_PROFILE_S * FS))


def measure(x_a, x_b, p):
    result = crqa.crqa(x_a, x_b, p["m"], p["tau"], p["radius"], min_line=min_line, max_lag=band)
    lags, profile = crqa.recurrence_profile(x_a, x_b, p["m"], p["tau"], p["radius"], profile_lag)
    return {"rec": result["rec"], "det": result["det"], "mean_line_s": result["mean_line"] / FS,
            "max_line_s": result["max_line"] / FS}, lags, profile


results, profiles = [], []
for s in segments:  # real pairs
    p = params[s["area"]]
    values, lags, profile = measure(s["x_a"], s["x_b"], p)
    info = {k: v for k, v in s.items() if not k.startswith("x_")}
    results.append({**info, **values, "surrogate": False})
    profiles.append({"area": s["area"], "participant_a": s["participant_a"], "participant_b": s["participant_b"],
                     "role": s["role"], "condition": s["condition"], "round": s["round"], "trial": s["trial"],
                     "surrogate": False, "profile": profile})

for (a, b, area), group in itertools.groupby(sorted(segments, key=lambda s: (s["participant_a"], s["participant_b"], s["area"])),
                                             key=lambda s: (s["participant_a"], s["participant_b"], s["area"])):
    group = list(group)
    for condition in ("joint", "individual"):
        same = [s for s in group if s["condition"] == condition]
        for si, sj in itertools.permutations(same, 2):  # A's trial i, B's trial j != i
            n = min(len(si["x_a"]), len(sj["x_b"]))
            values, lags, profile = measure(si["x_a"][:n], sj["x_b"][:n], params[area])
            results.append({"participant_a": a, "participant_b": b, "area": area, "condition": condition,
                            "round": si["round"], "trial": si["trial"], "round_b": sj["round"], "trial_b": sj["trial"],
                            "role": "surrogate", "n_samples": n, "duration_s": n / FS,
                            "bad_fraction_a": si["bad_fraction_a"], "bad_fraction_b": sj["bad_fraction_b"],
                            **values, "surrogate": True})
            profiles.append({"area": area, "participant_a": a, "participant_b": b, "role": "surrogate",
                             "condition": condition, "round": si["round"], "trial": si["trial"], "surrogate": True,
                             "profile": profile})

results = pd.DataFrame(results)
print(f"{(~results['surrogate']).sum()} real and {results['surrogate'].sum()} surrogate windows")

## 6. Save

- `crqa_<session>.csv`: one row per pair × area × window (real and surrogate), with the components, the role, the window (session time), the bad share, the parameters and the measures. The components are in every row, so a result always says which spatial filters it came from.
- `crqa_profiles_<session>.csv`: the diagonal profile (%REC per lag) of every real window, and the mean surrogate profile per pair, area and condition.
- `crqa_parameters_<session>.csv` (step 4).

In [ ]:
for col, key in (("ica", "ica"), ("component", "component"), ("band", "band")):
    results[f"{col}_a"] = [envelopes[(s, ar)][key] for s, ar in zip(results["participant_a"], results["area"])]
    results[f"{col}_b"] = [envelopes[(s, ar)][key] for s, ar in zip(results["participant_b"], results["area"])]
results["pair"] = results["participant_a"] + "_" + results["participant_b"]
results["session"] = SESSION
for k in ("m", "tau", "radius"):
    results[k] = [params[ar][k] for ar in results["area"]]
results["lag_band_s"], results["min_line_s"] = cfg.CRQA_LAG_BAND_S, cfg.CRQA_MIN_LINE_S
first = ["session", "area", "pair", "participant_a", "participant_b", "ica_a", "component_a", "ica_b", "component_b",
         "band_a", "condition", "round", "trial", "role", "surrogate", "rec", "det", "mean_line_s", "max_line_s"]
results = results[first + [c for c in results.columns if c not in first]]
results.to_csv(OUT / f"crqa_{SESSION}.csv", index=False)

lag_s = np.arange(-profile_lag, profile_lag + 1) / FS
prof = pd.DataFrame([{k: v for k, v in p.items() if k != "profile"} for p in profiles])
values = pd.DataFrame(np.vstack([p["profile"] for p in profiles]), columns=[f"{l:.2f}" for l in lag_s])
prof = pd.concat([prof, values], axis=1)
real = prof[~prof["surrogate"]]
sur = (prof[prof["surrogate"]].groupby(["area", "participant_a", "participant_b", "condition"])[list(values.columns)]
       .mean().reset_index().assign(role="surrogate", surrogate=True))
long = pd.concat([real, sur]).melt(id_vars=[c for c in prof.columns if c not in values.columns] ,
                                   value_vars=list(values.columns), var_name="lag_s", value_name="rec")
long["lag_s"] = long["lag_s"].astype(float)
long.to_csv(OUT / f"crqa_profiles_{SESSION}.csv", index=False)
print("saved to", OUT)

## 7. Overview

Mean %REC and %DET (lag band) per area and role, real and surrogate. This is only a description: the test is the mixed model over all pairs and sessions (`coupling ~ role + block + (1|session) + (1|pair) + ...`). With few pairs, read it as a check that the chain works.

The figure shows the mean diagonal profile per role: coupling would show as a peak near lag 0 above the surrogate line.

In [ ]:
summary = (results.groupby(["area", "role"])
           .agg(n=("rec", "size"), rec=("rec", "mean"), rec_sd=("rec", "std"), det=("det", "mean"), det_sd=("det", "std"),
                mean_line_s=("mean_line_s", "mean")).round(2))
display(summary)

fig, axes = plt.subplots(1, len(areas), figsize=(6 * len(areas), 3.5), squeeze=False)
for ax, area in zip(axes[0], areas):
    sub = long[long["area"] == area]
    for r, g in sub.groupby("role"):
        mean = g.groupby("lag_s")["rec"].mean()
        ax.plot(mean.index, mean.values, label=r, ls="--" if r == "surrogate" else "-")
    ax.axvspan(-cfg.CRQA_LAG_BAND_S, cfg.CRQA_LAG_BAND_S, color="0.9", zorder=0)
    ax.set(title=f"{SESSION} {area}", xlabel="lag (s): positive = B follows A", ylabel="%REC")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / f"crqa_profiles_{SESSION}.png", dpi=100)